# Clasificación Automática de Exoplanetas Kepler
Este notebook contiene el análisis de datos, preprocesamiento y entrenamiento del modelo de Machine Learning para clasificar Objetos de Interés de Kepler (KOI) en exoplanetas confirmados o falsos positivos.

### Paso 1: Importación de Librerías
Importamos las librerías necesarias para la manipulación de datos, división de conjuntos de datos, normalización, modelado, métricas y guardado del modelo.

In [1]:
# Importación de librerías base para análisis numérico y manipulación de datos
import pandas as pd
import numpy as np

# Importación de utilidades de scikit-learn para preprocesamiento y entrenamiento
from sklearn.model_selection import train_test_split  # Para dividir los datos en entrenamiento y prueba
from sklearn.preprocessing import StandardScaler      # Para normalizar las características continuas
from sklearn.ensemble import RandomForestClassifier     # Algoritmo de clasificación de ensamble
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score # Métricas de evaluación
import joblib # Para guardar y exportar el modelo y el escalador entrenados

ModuleNotFoundError: No module named 'pandas'

### Paso 2: Carga del Catálogo Acumulativo de Kepler
Cargamos el dataset oficial de avistamientos de Kepler en un DataFrame de pandas.

In [ ]:
# Cargar los datos acumulativos en formato CSV en un DataFrame de pandas
df = pd.read_csv("model/cumulative.csv")

### Paso 3: Exploración Inicial de los Datos
Inspeccionamos las primeras filas de la tabla para entender sus columnas y valores.

In [ ]:
# Mostrar las primeras 5 filas para una primera vista rápida de los datos
df.head()

NameError: name 'df' is not defined

### Paso 4: Limpieza y Filtrado de Clases Objetivo
Filtramos la variable objetivo (`koi_disposition`) para conservar solo las observaciones con confirmación definitiva: `CONFIRMED` y `FALSE POSITIVE`. Descartamos los registros marcados como `CANDIDATE` ya que aún no tienen una confirmación oficial y no nos servirían para el aprendizaje supervisado.

In [ ]:
# Filtrar el DataFrame original para conservar únicamente registros confirmados o falsos positivos conocidos
df = df[df['koi_disposition'].isin(['CONFIRMED', 'FALSE POSITIVE'])]

### Paso 5: Codificación Binaria de la Variable Objetivo
Mapeamos las etiquetas de texto a números binarios para que el modelo pueda procesarlos (1 para planetas confirmados, 0 para falsos positivos).


In [ ]:
# Crear la columna 'es_planeta' mapeando CONFIRMED a 1 y FALSE POSITIVE a 0
df['es_planeta'] = df['koi_disposition'].map({'CONFIRMED': 1, 'FALSE POSITIVE': 0})

### Paso 6: Selección de Características (Features)
Seleccionamos las características estelares, de coordenadas espaciales y banderas de falsos positivos más relevantes del catálogo. Cumplimos con el objetivo de utilizar más de 6 columnas de datos.

In [ ]:
# Lista de variables estelares y banderas predictivas que se ingresarán al modelo
columnas_importantes = [
    'koi_fpflag_nt',  # Anomalía: Bandera de curva de luz no compatible con tránsito planetario regular
    'koi_fpflag_ss',  # Anomalía: Bandera de eclipse binario o co-tránsito estelar de fondo detectado
    'koi_fpflag_co',  # Anomalía: Bandera de desplazamiento del centroide lumínico durante el tránsito
    'koi_slogg',      # Característica física: Gravedad en la superficie de la estrella (log(g))
    'koi_srad',       # Característica física: Radio estelar medido en múltiplos del radio solar
    'ra',             # Coordenada celeste: Ascensión Recta (longitud celeste)
    'dec',            # Coordenada celeste: Declinación (latitud celeste)
    'koi_kepmag',     # Brillo estelar aparente medido por el telescopio espacial Kepler
    'es_planeta'      # Variable objetivo (Clasificación real del catálogo)
]